# 03 · Clean, decide, reconcile

**Produces:** `data/clean/bookings_clean.csv.gz`, `data/clean/collections_clean.csv.gz`, `data/clean/patients_clean.csv.gz`, `data/clean/invoices_clean.csv.gz`, `data/clean/payments_clean.csv.gz`, and rows in the decisions log for every rule applied.

**Needs:** notebook 02 run.

Every cleaning act here is a decision with a reason, a row count and an effect on the headline. Input equals clean plus rejected after every step. Joins are checked before and after on row count, and a join that grows is explained or reverted.

## Change log

| Date | Change | Cells | Effect |
|---|---|---|---|
| 2026-10-24 | **Fix 2:** bookings are joined to collections on `coll_key` (the original booking id for a rescheduled nova visit) instead of `booking_id` | section 2, the LEFT JOIN cell (explained in the 🔧 cell just above it) | `left_only` 18,269 → 16,820. 1,449 rescheduled visits get their real collection status. Bookings and tests are unchanged. |
| 2026-10-24 | **Fix 2, part b:** the clock-skew check compares each collection with the slot it belongs to (the original slot for a nova reschedule) | section 2, the clock-skew cell (explained in the 🔧 cell just above it) | Without it, fix 2 would have flagged about 970 valid collection times as clock skew (1,905 instead of about 936). |

Outputs below were regenerated after the change. Notebooks 04, 05 and 06 were re-run on the new clean tables.

## CONFIG

In [1]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "<LETTER>",
    "group": "<N>",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

2.3.3 2.3.5


In [2]:
bookings = pd.read_csv(CONFIG["clean"] / "bookings.csv.gz", parse_dates=["created_at", "slot_at"], dtype={"booking_id": "string", "patient_id": "string", "rescheduled_from": "string", "status_at_export": "string"})
HEADLINE = lambda df: f"{len(df):,} bookings, {int(df['n_tests'].sum()):,} tests"
n0 = len(bookings)
print(HEADLINE(bookings))

564,879 bookings, 1,307,260 tests


## 1 · DO: the identity rule for a booking

Nova records a rescheduled visit as a new booking that points at the old one, and marks the old one `rescheduled`. Legacy overwrote the slot in place. If the question is "how many visits happened", a rescheduled pair is one visit. Write the rule, apply it, log it.

In [3]:
# status_at_export is NA for every legacy row. A nullable boolean treats NA as False only if you say so; left alone,
# ~NA is NA and the legacy rows vanish from the filter. The trap is worth seeing once, so it is stated here.
is_superseded = bookings["status_at_export"].eq("rescheduled").fillna(False).astype(bool)
kept = bookings[~is_superseded].copy()
rejected = bookings[is_superseded]
checks.reconciles("drop superseded nova records", n0, len(kept), len(rejected))
print("after:", HEADLINE(kept))
# logs.decision("bookings", "nova writes a new record on reschedule and marks the old one", "drop the superseded original",
#               "one visit, one booking; legacy overwrote in place so this makes the systems comparable",
#               rows_affected=len(rejected), effect_on_headline=f"{n0:,} -> {len(kept):,} bookings", owner="<name>", n=2)
bookings = kept

PASS  drop superseded nova records: input = kept + rejected   564,879 = 563,380 + 1,499
after: 563,380 bookings, 1,303,719 tests


> **Interview angle.** What is `pd.NA`, and what happens to `~series.eq('x')` when the series holds NA?
>
> *What a good answer covers:* pd.NA is the missing marker for nullable dtypes and propagates through comparisons, so eq gives NA, the negation stays NA, and a boolean filter drops those rows silently. fillna(False) before negating, or compare with notna() explicitly.

> **Interview angle.** Two systems record the same business event in different shapes. What is the general method for making them comparable, and what is the trap?
>
> *What a good answer covers:* Trace one real event through both systems by hand and write down how each represents it; then write a rule that maps both to one definition of the event; then apply it and check the count before and after. The trap is comparing totals without the rule and attributing the difference to the business.

## 2 · DO: collections, double scans, and the join that must not grow

In [4]:
coll_files = sorted((CONFIG["raw"] / "collections").glob("*.csv.gz"))
coll = loaders.read_many(coll_files, "legacy", workers=1) if False else pd.concat([pd.read_csv(p, dtype={"booking_id": "string"}) for p in coll_files], ignore_index=True)
coll["collected_at"] = pd.to_datetime(coll["collected_at"], errors="coerce")
n_in = len(coll)
dups = coll.duplicated(subset=["booking_id", "status", "collected_at"], keep="first")
coll_clean = coll[~dups].copy()
checks.reconciles("collections exact duplicates (double scans)", n_in, len(coll_clean), int(dups.sum()))
checks.check("one collection row per booking", coll_clean["booking_id"].is_unique, f"{(~coll_clean['booking_id'].is_unique)} duplicates remain" if not coll_clean["booking_id"].is_unique else "")

PASS  collections exact duplicates (double scans): input = kept + rejected   552,492 = 549,598 + 2,894


PASS  one collection row per booking


True

### 🔧 Change applied on 2026-10-24: fix 2, collections joined on the id the collection app used

**What was wrong.** When a patient reschedules in **nova** (Chennai and Jaipur from April 2026), nova writes a **new** booking (`rescheduled_from = <old id>`) and marks the old one `rescheduled`. Section 1 above drops the old record, which is correct: one visit, one booking. But the **phlebotomy app records the collection against the original booking id**. So after the drop:
- the collection row points at a booking that is no longer in the table, and
- the surviving new booking has no collection row, so it came out `left_only` and was labelled **`cancelled_or_missing`**.

Before the fix, **all 1,500** rescheduled nova visits looked cancelled, although **1,449** of them have a collection record (1,239 collected, 210 no-show). That inflated Chennai and Jaipur's cancellation rate in April–June 2026 and understated any count of collected bookings in exactly the two cities brief B asks about.

**What changed.** We build a join key, `coll_key`: the original id (`rescheduled_from`) for a nova reschedule, and the booking's own id for everything else. Bookings are joined to collections on that key. Three guards stay in place:
1. a check that `coll_key` is unique, so no two bookings can claim the same collection;
2. `validate="one_to_one"`, so pandas raises instead of fanning out;
3. `checks.join_did_not_fan_out`, so the row count stays 563,380.

`coll_key` is dropped after the join, so the clean table keeps its original columns.

**Expected effect** (checked on the clean data before the change): `left_only` falls from **18,269 to 16,820**, and the reschedules become 1,239 collected, 210 no-show and 51 genuinely unmatched. Booking and test counts are unchanged; this rule only changes the `status`, `collected_at` and `phlebotomist_id` columns.

**For the decisions log:** rule "collections join on original id for nova reschedules", 1,449 rows re-labelled, no effect on the test headline, reversible. The exact change in collected and no-show counts is computed in notebook 07, section 2.

In [5]:
# LEFT join bookings to collections. The row count must not grow. validate= makes pandas raise rather than lie.
# CHANGE (fix 2, 2026-10-24): the phlebotomy app records a rescheduled nova visit against the ORIGINAL booking id.
# Join on the id the collection app used: rescheduled_from for a nova reschedule, booking_id for everything else.
bookings["coll_key"] = bookings["rescheduled_from"].fillna(bookings["booking_id"])
checks.check("collection key is unique per booking (no two bookings claim one collection)",
             bookings["coll_key"].is_unique, f"{int(bookings['coll_key'].duplicated().sum())} repeated keys")
coll_side = coll_clean[["booking_id", "status", "collected_at", "phlebotomist_id"]].rename(columns={"booking_id": "coll_key"})
try:
    joined = bookings.merge(coll_side, on="coll_key", how="left", validate="one_to_one", indicator=True)
except pd.errors.MergeError as err:
    print("MergeError:", err)
    joined = bookings.merge(coll_side.drop_duplicates("coll_key"), on="coll_key", how="left", validate="one_to_one", indicator=True)
checks.join_did_not_fan_out("bookings LEFT JOIN collections on coll_key", bookings, joined, "booking_id")
resched = joined["rescheduled_from"].notna()
print(f"rescheduled nova visits matched to a collection through their original id: "
      f"{int((resched & joined['_merge'].eq('both')).sum()):,} of {int(resched.sum()):,}")
# logs.decision("bookings", "collection app records a nova reschedule against the original booking id",
#               "join collections on rescheduled_from for reschedules", "one visit keeps its collection after the identity rule",
#               rows_affected=int((resched & joined['_merge'].eq('both')).sum()), effect_on_headline="none on tests; status only", owner="<name>", n=3)
joined = joined.drop(columns="coll_key")
joined["_merge"].value_counts()


PASS  collection key is unique per booking (no two bookings claim one collection)   0 repeated keys


PASS  bookings LEFT JOIN collections on coll_key: join kept the left row count   left 563,380, joined 563,380, rows on repeated keys 0
rescheduled nova visits matched to a collection through their original id: 1,449 of 1,500


_merge
both          546560
left_only      16820
right_only         0
Name: count, dtype: int64

### SEE: what `left_only` means

A booking with no collection row is a cancellation (the phlebotomy app does not record cancelled bookings) or a booking in the future at export time. Decide which, and write it down. A row that is `left_only` and dated months ago is a question for the data team, and a caveat on your slide.

### 🔧 Change applied on 2026-10-24: fix 2, part b, the clock-skew check uses the slot the collection belongs to

**Why this was needed.** Once fix 2 matched 1,449 rescheduled visits to their collection rows, the clock-skew check below flagged **1,905** collections instead of 936. We looked at the timestamps, and they're not clock errors:
- **99.7%** of the rescheduled visits' collections happen within 6 hours of the **original** slot. The median is 0.57 hours after it, exactly the pattern of ordinary bookings.
- **None** happens within 6 hours of the **new** slot, which is 1 to 4 days after the original.

So the collection belongs to the original slot. Comparing it with the new slot made about 970 valid timestamps look "collected days before the slot", and the rule then blanked them.

**What changed.** The reference slot for the skew check is now the original booking's slot for a nova reschedule (looked up from `rejected`, the superseded originals set aside in section 1), and the booking's own slot for everything else. The tolerance (one day) is unchanged.

**Expected effect.** The clock-skew count returns to about the pre-fix level (936), and the rescheduled visits keep their real collection times.

**An open question for the group (not changed here).** If the sample was taken at the original slot time, the visit probably happened at the original slot, and the identity rule's choice to keep the *new* record's `slot_at` puts some visits on the wrong date. Notebook 07, section 3, measures how much that moves the half-year counts, so the group can decide whether to keep the original slot instead.

In [6]:
joined["status"] = joined["status"].fillna("cancelled_or_missing")
# CHANGE (fix 2, part b, 2026-10-24): compare each collection with the slot it belongs to. For a rescheduled nova visit the
# collection app logs the sample against the ORIGINAL booking, at the ORIGINAL slot time, so the reference is the original slot
# (kept in `rejected` from section 1). Comparing with the new slot would flag about 970 valid timestamps as clock skew.
orig_slot = rejected.set_index("booking_id")["slot_at"]
ref_slot = joined["rescheduled_from"].map(orig_slot).fillna(joined["slot_at"])
skew = joined["collected_at"] < ref_slot - pd.Timedelta(days=1)
print("collected more than a day before the slot (clock skew?):", int(skew.sum()))
joined.loc[skew, "collected_at"] = pd.NaT       # keep the row, blank the timestamp, flag it
joined["flag_clock_skew"] = skew
bookings = joined.drop(columns="_merge")

> **Interview angle.** Explain `how='left'`, `validate='one_to_one'` and `indicator=True` to someone who has only used SQL.
>
> *What a good answer covers:* LEFT keeps every left row whether or not it matches, like SQL LEFT JOIN. validate asserts the key's cardinality on each side and raises if it is violated, which SQL does not do for you. indicator adds a column saying which side each row came from, which is the SQL anti-join check (WHERE right.key IS NULL) made explicit.

## 3 · DO: patients, and the duplicate that looks different

In [7]:
pat = pd.read_csv(CONFIG["raw"] / "patients" / "patients.csv.gz", dtype="string")
pat["phone_norm"] = loaders.normalise_phone(pat["phone"])
pat["name_norm"] = pat["patient_name"].str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
grp = pat.groupby(["phone_norm", "name_norm", "dob"]).ngroup()
pat["person_key"] = grp
dup_mask = pat.duplicated(subset=["person_key"], keep="first")
print(f"{len(pat):,} patient rows, {pat['person_key'].nunique():,} distinct people by phone+name+dob, {int(dup_mask.sum()):,} duplicate rows")
bad_dob = pat["dob"].eq("1900-01-01")
print("placeholder dob rows:", int(bad_dob.sum()))
pat_clean = pat[~dup_mask].copy()
checks.reconciles("patients de-duplicated", len(pat), len(pat_clean), int(dup_mask.sum()))
# map every duplicate id to its surviving id so bookings can be re-keyed to people, not to ids
id_map = pat.merge(pat_clean[["person_key", "patient_id"]].rename(columns={"patient_id": "person_id"}), on="person_key")[["patient_id", "person_id"]]
bookings = bookings.merge(id_map, on="patient_id", how="left", validate="many_to_one")
checks.check("every booking patient id maps to a person", bool(bookings["person_id"].notna().all()), int(bookings["person_id"].isna().sum()))

309,217 patient rows, 303,153 distinct people by phone+name+dob, 6,064 duplicate rows
placeholder dob rows: 1492
PASS  patients de-duplicated: input = kept + rejected   309,217 = 303,153 + 6,064


PASS  every booking patient id maps to a person   0


True

### The decision here is not technical

Merging two patient rows says two records are one human. For a growth analysis the cost of being wrong is a slightly wrong frequency figure. For a results system it would be a wrong report sent to the wrong person. Same rule, different stakes. Write the stakes in the decisions log beside the rule.

## 4 · DO: invoices and payments, two id families

In [8]:
inv = pd.concat([pd.read_csv(p, dtype="string") for p in sorted((CONFIG["raw"] / "invoices").glob("*.csv.gz"))], ignore_index=True)
inv["amount_rs"] = inv["amount"].str.replace(",", "", regex=False).astype(float)
inv["invoice_date"] = pd.to_datetime(inv["invoice_date"])
pay = pd.concat([pd.read_csv(p, dtype={"notes_invoice_ref": "string", "settlement_id": "string"}) for p in sorted((CONFIG["raw"] / "payments").glob("gateway_*.csv.gz"))], ignore_index=True)
pay["amount_rs"] = pay["amount_paise"] / 100
pay["created_at"] = pd.to_datetime(pay["created_at_epoch"], unit="s") + pd.Timedelta(hours=5, minutes=30)
pay["invoice_id"] = loaders.normalise_invoice_ref(pay["notes_invoice_ref"])
print(f"invoices {len(inv):,} for Rs {inv['amount_rs'].sum():,.0f};  payment rows {len(pay):,} ({(pay.status=='captured').sum():,} captured, {(pay.status=='failed').sum():,} failed)")
checks.check("invoice ids unique", inv["invoice_id"].is_unique)

invoices 470,776 for Rs 737,020,069;  payment rows 473,507 (455,469 captured, 18,038 failed)
PASS  invoice ids unique


True

In [9]:
captured = pay[pay["status"] == "captured"]
m = inv.merge(captured[["invoice_id", "payment_id", "amount_rs", "created_at"]].rename(columns={"amount_rs": "paid_rs"}), on="invoice_id", how="outer", indicator=True, validate="one_to_many")
print(m["_merge"].value_counts())
multi = m[m["_merge"] == "both"].groupby("invoice_id").size()
print("invoices with more than one captured payment:", int((multi > 1).sum()))

_merge
both          450881
left_only      21820
right_only      4588
Name: count, dtype: int64


invoices with more than one captured payment: 1925


> **Interview angle.** Your LEFT join of invoices to payments grew the row count. Name three possible causes and the check that separates them.
>
> *What a good answer covers:* Retries (same invoice, a failed and a captured row): filter by status before joining. Genuine double charges (two captured rows): group by invoice and count captured. A many-to-many on a non-unique key such as amount and date: check key uniqueness on the right side with validate='one_to_many' or a groupby count before the join.

## 5 · SUM: write the clean tables and log every rule

In [10]:
bookings.to_csv(CONFIG["clean"] / "bookings_clean.csv.gz", index=False, compression="gzip")
coll_clean.to_csv(CONFIG["clean"] / "collections_clean.csv.gz", index=False, compression="gzip")
pat_clean.to_csv(CONFIG["clean"] / "patients_clean.csv.gz", index=False, compression="gzip")
inv.to_csv(CONFIG["clean"] / "invoices_clean.csv.gz", index=False, compression="gzip")
pay.to_csv(CONFIG["clean"] / "payments_clean.csv.gz", index=False, compression="gzip")
print("headline after cleaning:", HEADLINE(bookings))
checks.summary()

headline after cleaning: 563,380 bookings, 1,303,719 tests
8 of 8 checks passed


,check,passed,detail
0,drop superseded nova records: input = kept + r...,True,"564,879 = 563,380 + 1,499"
1,collections exact duplicates (double scans): i...,True,"552,492 = 549,598 + 2,894"
2,one collection row per booking,True,
3,collection key is unique per booking (no two b...,True,0 repeated keys
4,bookings LEFT JOIN collections on coll_key: jo...,True,"left 563,380, joined 563,380, rows on repeated..."
5,patients de-duplicated: input = kept + rejected,True,"309,217 = 303,153 + 6,064"
6,every booking patient id maps to a person,True,0
7,invoice ids unique,True,


## Before you move on

- List every rule you applied, in order, with the headline number before and after each. That list is `docs/decisions-log.md`; if it is not there yet, stop and write it.
- Which rule moved the headline most? That rule is on slide 5.
- Which rule would the finance controller challenge first?